In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
import torch
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, recall_score
from sklearn.model_selection import train_test_split
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import fbeta_score
import numpy as np
from collections import Counter
import itertools
import json
import os
import pickle
import warnings
from dataclasses import dataclass, field, asdict
from typing import Callable

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [17]:
df = pd.read_csv('./data/bolivia_train_cleaned.csv')

In [18]:
df.head()

,bank_code,bank_tier,client_segment,channel,card_brand,MTI,DE39_response_code,amount_usd,is_international,distance_from_home_km,approved,client_baseline_amount,is_fraud,is_night,time_since_last_txn_min,is_online,is_foreign_currency,mcc_fraud_rate
0,0,0,0,3,0,100,0,110.83,0,9.6,1,833.88,0,1,9999.0,0,0.0,0.009920
1,0,0,1,3,0,100,0,459.59,0,202.0,1,2392.95,0,1,9999.0,0,0.0,0.060447
2,0,0,2,3,1,100,0,87.62,1,5633.0,1,760.48,0,1,9999.0,0,1.0,0.009504
3,0,0,3,1,1,100,0,2159.82,0,3.2,1,1770.28,0,1,9999.0,1,0.0,0.045039
4,0,0,3,1,1,100,0,131.76,0,9.4,1,660.09,0,1,9999.0,1,0.0,0.105871


In [19]:
df.columns

Index(['bank_code', 'bank_tier', 'client_segment', 'channel', 'card_brand',
       'MTI', 'DE39_response_code', 'amount_usd', 'is_international',
       'distance_from_home_km', 'approved', 'client_baseline_amount',
       'is_fraud', 'is_night', 'time_since_last_txn_min', 'is_online',
       'is_foreign_currency', 'mcc_fraud_rate'],
      dtype='str')

In [20]:
df.shape

(83836, 18)

In [21]:
X_train = df.drop('is_fraud', axis = 1)

In [22]:
Y_train = df['is_fraud']

In [23]:
bool_cols = X_train.select_dtypes(include=['bool']).columns

X_train[bool_cols] = X_train[bool_cols].astype(int)

In [25]:
test = pd.read_csv('./data/bolivia_test_cleaned.csv')

In [26]:
x_test = test.drop('is_fraud', axis = 1)
y_test = test['is_fraud']

In [27]:
bool_cols = x_test.select_dtypes(include=['bool']).columns

x_test[bool_cols] = x_test[bool_cols].astype(int)

In [28]:
classes = Y_train.value_counts()

In [29]:
classes

is_fraud
0    79645
1     4191
Name: count, dtype: int64

In [30]:
new_classes = y_test.value_counts()

In [33]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# --- Scale pos weight con ambos conjuntos ---
n_pos = (Y_train == 1).sum() + (y_test == 1).sum()
n_neg = (Y_train == 0).sum() + (y_test == 0).sum()
scale_pos_weight = n_neg / n_pos
print(f"scale_pos_weight: {scale_pos_weight:.2f} (pos={n_pos}, neg={n_neg})")

# --- Validación desde train (15%) ---
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, Y_train,
    test_size=0.15,
    stratify=Y_train,   # mantiene proporción de fraude
    random_state=42
)
print(f"X_tr: {X_tr.shape} | X_val: {X_val.shape}")

# --- Modelo base anti-overfitting ---
params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'min_child_samples': 50,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'scale_pos_weight': scale_pos_weight,
    'verbose': -1,
    'random_state': 42,
}

model = lgb.LGBMClassifier(
    **params,
    n_estimators=500,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
    ]
)

# --- Evaluación rápida ---
from sklearn.metrics import roc_auc_score, classification_report

y_proba = model.predict_proba(x_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"\nAUC en test: {auc:.4f}")
print(classification_report(y_test, (y_proba >= 0.5).astype(int)))

scale_pos_weight: 19.33 (pos=4919, neg=95084)
X_tr: (71260, 17) | X_val: (12576, 17)
Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.921124
[100]	valid_0's auc: 0.921643
[150]	valid_0's auc: 0.922459
Early stopping, best iteration is:
[147]	valid_0's auc: 0.922508

AUC en test: 0.9070
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     15439
           1       0.82      0.78      0.80       728

    accuracy                           0.98     16167
   macro avg       0.90      0.89      0.89     16167
weighted avg       0.98      0.98      0.98     16167



In [37]:
import optuna
from sklearn.metrics import roc_auc_score, recall_score
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbose': -1,
        'random_state': 42,
        'scale_pos_weight': scale_pos_weight,
        'num_leaves':        trial.suggest_int('num_leaves', 20, 80),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    model = lgb.LGBMClassifier(**params, n_estimators=1000)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ]
    )

    y_proba = model.predict_proba(X_val)[:, 1]
    
    # --- Buscar threshold óptimo en val ---
    best_score = 0
    for t in np.linspace(0.1, 0.9, 100):
        y_hat = (y_proba >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_val == 1))
        fp = np.sum((y_hat == 1) & (y_val == 0))
        fn = np.sum((y_hat == 0) & (y_val == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # F-beta: beta>1 favorece recall, beta<1 favorece precision
        # beta=1 → F1 puro
        beta = 1
        f_beta = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall + 1e-9)
        
        if f_beta > best_score:
            best_score = f_beta

    return best_score


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

[I 2026-06-03 02:55:19,936] A new study created in memory with name: no-name-53101042-61f5-4c8f-a87f-5923ea729e82
Best trial: 0. Best value: 0.846878:   1%|          | 1/100 [00:00<00:42,  2.35it/s]

[I 2026-06-03 02:55:20,363] Trial 0 finished with value: 0.8468776727278766 and parameters: {'num_leaves': 23, 'max_depth': 9, 'min_child_samples': 131, 'learning_rate': 0.016778087173252496, 'subsample': 0.8278610665527486, 'colsample_bytree': 0.850444460268388, 'reg_alpha': 1.7865151989165773, 'reg_lambda': 0.1944661073275087}. Best is trial 0 with value: 0.8468776727278766.


Best trial: 1. Best value: 0.848896:   2%|▏         | 2/100 [00:00<00:32,  3.02it/s]

[I 2026-06-03 02:55:20,628] Trial 1 finished with value: 0.8488964341372806 and parameters: {'num_leaves': 52, 'max_depth': 6, 'min_child_samples': 143, 'learning_rate': 0.08983090221160105, 'subsample': 0.7773808203885744, 'colsample_bytree': 0.6185703117309486, 'reg_alpha': 0.00032049693613501654, 'reg_lambda': 0.0004068906596330761}. Best is trial 1 with value: 0.8488964341372806.


Best trial: 2. Best value: 0.850594:   3%|▎         | 3/100 [00:01<00:36,  2.68it/s]

...

[I 2026-06-03 02:56:17,579] Trial 97 finished with value: 0.8716216211235747 and parameters: {'num_leaves': 56, 'max_depth': 10, 'min_child_samples': 36, 'learning_rate': 0.03352062254546505, 'subsample': 0.9048170838242054, 'colsample_bytree': 0.6009421655161297, 'reg_alpha': 0.0017009729229196257, 'reg_lambda': 0.0033487670133270377}. Best is trial 81 with value: 0.8736217128186147.


Best trial: 81. Best value: 0.873622:  99%|█████████▉| 99/100 [00:58<00:00,  1.17it/s]

[I 2026-06-03 02:56:18,327] Trial 98 finished with value: 0.8609941022819068 and parameters: {'num_leaves': 63, 'max_depth': 10, 'min_child_samples': 35, 'learning_rate': 0.02890704016884679, 'subsample': 0.8953668766133868, 'colsample_bytree': 0.6267502886487062, 'reg_alpha': 0.0013127455878010604, 'reg_lambda': 0.0033635222143068425}. Best is trial 81 with value: 0.8736217128186147.


Best trial: 81. Best value: 0.873622: 100%|██████████| 100/100 [00:58<00:00,  1.70it/s]

[I 2026-06-03 02:56:18,922] Trial 99 finished with value: 0.8631402178055285 and parameters: {'num_leaves': 56, 'max_depth': 10, 'min_child_samples': 46, 'learning_rate': 0.03272533035486269, 'subsample': 0.8623973632919313, 'colsample_bytree': 0.6010645229749617, 'reg_alpha': 0.01310466029096628, 'reg_lambda': 0.0009282029171995398}. Best is trial 81 with value: 0.8736217128186147.


In [38]:
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'random_state': 42,
    'scale_pos_weight': scale_pos_weight,
    **study.best_params
}

best_model = lgb.LGBMClassifier(**best_params, n_estimators=500)
best_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
    ]
)

y_proba = best_model.predict_proba(x_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"\nAUC en test: {auc:.4f}")
print(classification_report(y_test, (y_proba >= 0.5).astype(int)))

Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.915736
[100]	valid_0's auc: 0.917206
[150]	valid_0's auc: 0.918829
[200]	valid_0's auc: 0.918928
Early stopping, best iteration is:
[189]	valid_0's auc: 0.919144

AUC en test: 0.9062
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     15439
           1       0.84      0.78      0.81       728

    accuracy                           0.98     16167
   macro avg       0.91      0.89      0.90     16167
weighted avg       0.98      0.98      0.98     16167



In [41]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def feval_recall_minus_fpratio(y_true, y_pred_raw):
    y_proba = sigmoid(y_pred_raw)
    best_score = -np.inf

    for t in np.linspace(0.1, 0.9, 100):
        y_hat = (y_proba >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_true == 1))
        fp = np.sum((y_hat == 1) & (y_true == 0))
        fn = np.sum((y_hat == 0) & (y_true == 1))
        recall   = tp / (tp + fn + 1e-9)
        fp_ratio = fp / (tp + fp + 1e-9)
        if recall >= 0.90:
            score = recall - 2 * fp_ratio
            if score > best_score:
                best_score = score

    return 'recall_minus_2xfpr', best_score, True


def feval_fbeta_precision(y_true, y_pred_raw):
    y_proba = sigmoid(y_pred_raw)
    best_score = -np.inf
    beta = 0.5

    for t in np.linspace(0.1, 0.9, 100):
        y_hat = (y_proba >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_true == 1))
        fp = np.sum((y_hat == 1) & (y_true == 0))
        fn = np.sum((y_hat == 0) & (y_true == 1))
        recall    = tp / (tp + fn + 1e-9)
        precision = tp / (tp + fp + 1e-9)
        if recall >= 0.90:
            f_beta = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall + 1e-9)
            if f_beta > best_score:
                best_score = f_beta

    return 'fbeta_0.5_recall90', best_score, True


def feval_precision_at_recall90(y_true, y_pred_raw):
    y_proba = sigmoid(y_pred_raw)
    best_score = -np.inf

    for t in np.linspace(0.1, 0.9, 100):
        y_hat = (y_proba >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_true == 1))
        fp = np.sum((y_hat == 1) & (y_true == 0))
        fn = np.sum((y_hat == 0) & (y_true == 1))
        recall    = tp / (tp + fn + 1e-9)
        precision = tp / (tp + fp + 1e-9)
        if recall >= 0.90:
            if precision > best_score:
                best_score = precision

    return 'precision_at_recall90', best_score, True

In [42]:
results = {}

for feval_fn in [feval_recall_minus_fpratio, feval_fbeta_precision, feval_precision_at_recall90]:
    model = lgb.LGBMClassifier(**best_params, n_estimators=1000)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
        eval_metric=feval_fn,
    )
    
    y_proba = model.predict_proba(x_test)[:, 1]
    
    # Buscar threshold óptimo en val
    best_t, best_recall, best_fp_ratio = 0.5, 0, 1
    for t in np.linspace(0.1, 0.9, 200):
        y_hat = (model.predict_proba(X_val)[:, 1] >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_val == 1))
        fp = np.sum((y_hat == 1) & (y_val == 0))
        fn = np.sum((y_hat == 0) & (y_val == 1))
        recall = tp / (tp + fn + 1e-9)
        fp_ratio = fp / (tp + fp + 1e-9)
        if recall >= 0.90 and fp_ratio < best_fp_ratio:
            best_fp_ratio = fp_ratio
            best_recall = recall
            best_t = t
    
    # Evaluar en test
    y_hat_test = (y_proba >= best_t).astype(int)
    tp = np.sum((y_hat_test == 1) & (y_test == 1))
    fp = np.sum((y_hat_test == 1) & (y_test == 0))
    fn = np.sum((y_hat_test == 0) & (y_test == 1))
    recall_test   = tp / (tp + fn + 1e-9)
    fp_ratio_test = fp / (tp + fp + 1e-9)
    auc_test      = roc_auc_score(y_test, y_proba)
    
    name = feval_fn.__name__
    results[name] = {
        'threshold': best_t,
        'recall':    recall_test,
        'fp_ratio':  fp_ratio_test,
        'auc':       auc_test,
    }
    print(f"\n{name}")
    print(f"  Threshold: {best_t:.3f} | Recall: {recall_test:.3f} | FP_ratio: {fp_ratio_test:.3f} | AUC: {auc_test:.4f}")

# Tabla comparativa
print("\n=== COMPARATIVA FEVALS ===")
print(f"{'Feval':<35} {'Threshold':>10} {'Recall':>8} {'FP_ratio':>10} {'AUC':>8}")
for name, r in results.items():
    print(f"{name:<35} {r['threshold']:>10.3f} {r['recall']:>8.3f} {r['fp_ratio']:>10.3f} {r['auc']:>8.4f}")


feval_recall_minus_fpratio
  Threshold: 0.144 | Recall: 0.894 | FP_ratio: 0.908 | AUC: 0.9065

feval_fbeta_precision
  Threshold: 0.144 | Recall: 0.894 | FP_ratio: 0.908 | AUC: 0.9065

feval_precision_at_recall90
  Threshold: 0.144 | Recall: 0.894 | FP_ratio: 0.908 | AUC: 0.9065

=== COMPARATIVA FEVALS ===
Feval                                Threshold   Recall   FP_ratio      AUC
feval_recall_minus_fpratio               0.144    0.894      0.908   0.9065
feval_fbeta_precision                    0.144    0.894      0.908   0.9065
feval_precision_at_recall90              0.144    0.894      0.908   0.9065
